<table style="margin: auto; background-color: white;">
    <tr>
      <td style="background-color: white;">
        <img src='https://drive.google.com/uc?export=view&id=1lgflViz1uefcvVW1iI57haB4M1bKsZtp' alt="drawing" width="200" />
      </td>
      <td style="background-color: white;">
        <img src='https://drive.google.com/uc?export=view&id=1R6PphT9jmd2vikODFPf6cW54QtZ29o2a' alt="drawing" width="200" />
      </td>
      <td style="background-color: white;">
        <img src='https://drive.google.com/uc?export=view&id=1lgflViz1uefcvVW1iI57haB4M1bKsZtp' alt="drawing" width="200" />
      </td>
      <td style="background-color: white;">
        <img src='https://drive.google.com/uc?export=view&id=1R6PphT9jmd2vikODFPf6cW54QtZ29o2a' alt="drawing" width="200" />
      </td>
      <td style="background-color: white;">
        <img src='https://drive.google.com/uc?export=view&id=1lgflViz1uefcvVW1iI57haB4M1bKsZtp' alt="drawing" width="200" />
      </td>
      <td style="background-color: white;">
        <img src='https://drive.google.com/uc?export=view&id=1R6PphT9jmd2vikODFPf6cW54QtZ29o2a' alt="drawing" width="200" />
      </td>
      <td style="background-color: white;">
        <img src='https://drive.google.com/uc?export=view&id=1lgflViz1uefcvVW1iI57haB4M1bKsZtp' alt="drawing" width="200" />
      </td>
    </tr>
</table>

# TUTORIAL 4 - FMNIST E SELEÇÃO DE CLIENTES

Bem-vindo! Neste tutorial você aprenderá sobre a interface de programação da plataforma **Flautim** e também como montar um experimento simples de classificação usando o dataset [FMNIST](https://huggingface.co/datasets/zalando-datasets/fashion_mnist) com **seleção de clientes**. 

É recomendado que você já esteja familiarizado com aprendizado federado e utilização da plataforma Flautim, tendo realizado algum dos outros tutoriais previamente.

O código desse tutorial pode ser acessado em: [clique aqui](./TUTORIAL_4/).


Vamos começar entendendo a interface de programação da **Flautim**, representada na figura abaixo. A **Flautim_api** é uma biblioteca modularizada que facilita a realização de experimentos de aprendizado de máquina, seja convencional/centralizado ou federado.

Todo projeto **Flautim** precisa herdar essa biblioteca, que contém submódulos específicos para diferentes tecnologias (por exemplo, submódulos para PyTorch, TensorFlow, etc). Neste tutorial usaremos o submódulo para PyTorch.

<div style="text-align: center;"> <table style="margin: auto;"> <tr> <td> <img src='https://drive.google.com/uc?export=view&id=1QOI4jWrwS979xhW_wlGzkPa2bMA-giuc' alt="Interface da plataforma Flautim" width="800" /> </td> </tr> </table> </div>


Dentro de cada submódulo existem três componentes principais (classes):

**1. Dataset:** é utilizado para representar os dados do experimento. Esta classe pode ser reutilizada em diversos experimentos e com diferentes modelos, sendo o componente mais versátil e reutilizável. Os usuários podem importar os dados de diversas fontes, como arquivos locais ou bases de dados online, desde que a classe Dataset seja herdada.

**2. Model:** representa qualquer conjunto de parâmetros treináveis dentro do projeto. Ela permite a aplicação de técnicas de aprendizado de máquina por meio de treinamento desses parâmetros. No caso de PyTorch, a classe herda a nn.Module, que define a estrutura e os parâmetros treináveis do modelo.

**3. Experiment:** define o ciclo de treinamento e validação. Existem dois tipos principais de experimentos: o experimento centralizado, que segue o fluxo
convencional de aprendizado de máquina, e o experimento federado, adaptado para
aprendizado federado. Esta classe inclui duas funções principais, um loop de
treinamento e um loop de validação, que realizam a atualização dos parâmetros e
cálculo das métricas de custo, respectivamente.

Além desses três componentes principais, há também um módulo chamado Common. Este módulo fornece acesso a classes essenciais para o gerenciamento de dados e monitoramento do treinamento.


Com essa visão geral, você está pronto para começar montar seus próprios experimentos. Vamos ao passo a passo!

### Passo 1: Criando o dataset que será usado no experimento

Um conjunto de dados no Flautim é acessado por um arquivo .py que deve conter uma classe que herda de Dataset.

**Exemplo: Implementando a Classe FMNISTDataset**

O código abaixo implementa uma classe FMNISTDataset utilizando o dataset FMNIST obtido pelo Hugging Face para resolver um problema de classificação.

In [ ]:
from flautim.pytorch.Dataset import Dataset
import torch
import copy

class TitanicDataset(Dataset):
    def __init__(self, file, **kwargs):
        super(TitanicDataset, self).__init__(name="TITANIC", **kwargs)

        self.features = file[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
                               'Title', 'FamilySize', 'IsAlone', 'Embarked', 'Deck']].values
        self.target = file['Survived'].values

        self.xdtype = torch.float32
        self.ydtype = torch.int64
        self.batch_size = 10
        self.shuffle = True
        self.num_workers = 1
        self.test_size = int(0.2 * len(self.features))

    def train(self):
        train = copy.deepcopy(self)
        train.features = self.features[:-self.test_size]
        train.target = self.target[:-self.test_size]
        return copy.deepcopy(train)

    def validation(self):
        test = copy.deepcopy(self)
        test.features = self.features[-self.test_size:]
        test.target = self.target[-self.test_size:]
        return copy.deepcopy(test)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return (torch.tensor(self.features[idx], dtype=torch.float32),
                torch.LongTensor([self.target[idx]]))

### Passo 2: Criando o modelo que será usado no experimento

Agora, vamos criar a classe que implementa o modelo. Essa classe deve herdar da classe Model.


**Exemplo: Implementando a Classe PoCModel_2HiddenLayers**

A classe ```PoCModel_2HiddenLayers``` implementa uma rede neural convolucional baseada na rede proposta do Paper Power-Of-Choice, com as seguintes camadas:
* Pre-processamento (flatten): a entrada (assumidamente um lote de imagens 28x28) é achatada em um vetor de 784 características
* Uma camada oculta totalmente conectada, com entrada de 784 neurônios e saída de 256 neurônios. Ativação ReLU.
* Uma camada oculta totalmente conectada, com entrada de 256 neurônios e saída de 128 neurônios. Ativação ReLU.
* Uma camada de saída totalmente conectada com número de neurônios igual ao número de classes do problema.

Essa classe pode ser incluída, por exemplo, em um arquivo MNISPoCModel_2HiddenLayers.py

In [ ]:
from flautim.pytorch.Model import Model
import torch

class TitanicModel(Model):
    def __init__(self, context, num_classes=2, **kwargs):
        super(TitanicModel, self).__init__(context, name="TITANIC-NN", **kwargs)

        self.c1 = torch.nn.Linear(11, 32)
        self.c2 = torch.nn.Linear(32, 16)
        self.c3 = torch.nn.Linear(16, num_classes)

    def forward(self, x):
        x = torch.relu(self.c1(x))
        x = torch.relu(self.c2(x))
        x = self.c3(x)
        return x

### Passo 3: Criando o experimento

Por fim, será criado o experimento, isto é, uma classe que implementa os loops de treinamento e validação do modelo PoCModel_2HiddenLayers no dataset FMNISTDataset. Para isso, precisamos criar dois arquivos .py, o run.py (que deve ter obrigatoriamente esse nome) e o .py responsável por implementar o experimento, descritos a seguir:

**1. Arquivo run.py:**

* Esse arquivo é o ponto de entrada de todo experimento Flautim, pois é ele
que deve iniciar a classe do experimento e também um modelo e um Dataset.

**2. Arquivo .py do experimento:**

* Esse arquivo deve conter uma classe que implemente os métodos de treinamento (training_loop) e validação (evaluation_loop) do modelo. Essa classe deve herdar da classe Experiment.

Esse tutorial cobrirá dois tipos de experimentos, um experimento centralizado e outro descentralizado. Portanto, o passo 3 será dividido entre esses dois cenários.

#### **Seleção de Clientes**
Ao considerar o funcionamento do aprendizado federado, onde os modelos locais dos clientes são enviados para o servidor central que utilizará uma estratégia de agregação (como FedAvg) para treinar o modelo global, a participação dos clientes é um fator importante para o desempenho do modelo. 

Ao utilizarmos, por exemplo, o [framework flower](https://flower.ai/docs/framework/#) sem definir uma seleção específica, a quantidade percentual de clientes definida no ```fraction_fit``` será usada para escolher **aleatoriamente** entre os clientes disponíveis na rodada. 
Essa entre os clientes é feita pela  ``` class SimpleClientManager (ClientManager) ```
em ``` (207) sampled_cids = random.sample(available_cids, num_clients) ``` 
e pode ser estudada em https://github.com/adap/flower/blob/main/framework/py/flwr/server/client_manager.py.

##### Definindo uma nova estratégia

Na literatura diversos autores já abordaram o tema, como [Cho et al.](https://proceedings.mlr.press/v151/jee-cho22a/jee-cho22a.pdf) que propõem Power-Of-Choice (POC), uma estrutura de seleção de clientes que direciona a seleção de clientes para clientes com maior perda local, permitindo uma convergência mais rápida e aumentamdo a eficiência da comunicação.

Para definirmos uma nova estratégia de seleção de clientes, usualmente estamos interresados em escolher de acordo com uma informação do treinamento atual (por exemplo a _loss_ para POC). Essas informações só estão disponíveis durante a etapa de treinamento (onde o metodo _fit_ é chamado), anterior a agregação do servidor. Assim, a solução adotada é definir uma nova estratégia extendendo uma existente (como FedAvg) e modificando o método ```configure_fit```.



#### Passo 3.1: Experimento federado
**Implementando a Classe FMNISTExperiment**

No código abaixo, criamos a classe FMNISTExperiment no modo federado com seus métodos training_loop e evaluation_loop para treinar e testar a rede neural. Esses métodos retornam o valor da função de perda e a acurácia de treinamento e de validação.

Repare que o método ```fit``` é redefinido para ter um tratamento extra da _loss_ local a ser utilizado na estratégia POC.

In [ ]:
from flautim.pytorch.centralized.Experiment import Experiment
import flautim as fl
import numpy as np
import torch

class TitanicExperiment(Experiment):
    def __init__(self, model, dataset, context, **kwargs):
        super(TitanicExperiment, self).__init__(model, dataset, context, **kwargs)

        self.criterion = torch.nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=0.01)
        self.epochs = kwargs.get('epochs', 30)

    def training_loop(self, data_loader):
        self.model.train()
        error_loss = 0.0
        yhat, y_real = [], []

        for X, y in data_loader:
            self.optimizer.zero_grad()
            outputs = self.model(X)
            loss = self.criterion(outputs, y)
            loss.backward()
            self.optimizer.step()

            error_loss += loss.cpu().item()
            _, predicted = torch.max(outputs.data, 1)
            yhat.append(predicted.detach().cpu())
            y_real.append(y.detach().cpu())

        yhat_np = torch.cat(yhat).numpy()
        y_real_np = torch.cat(y_real).numpy()

        accuracy   = self.metrics.ACCURACY(yhat_np, y_real_np)
        accuracy_2 = self.metrics.ACCURACY_2(yhat_np, y_real_np)
        error_loss = error_loss / len(data_loader)

        return float(error_loss), {'ACCURACY': accuracy, 'ACCURACY_2': accuracy_2}

    def validation_loop(self, data_loader):
        self.model.eval()
        error_loss = 0.0
        yhat, y_real = [], []

        with torch.no_grad():
            for X, y in data_loader:
                outputs = self.model(X)
                error_loss += self.criterion(outputs, y).item()
                _, predicted = torch.max(outputs.data, 1)
                yhat.append(predicted.detach().cpu())
                y_real.append(y.detach().cpu())

        yhat_np = torch.cat(yhat).numpy()
        y_real_np = torch.cat(y_real).numpy()

        accuracy   = self.metrics.ACCURACY(yhat_np, y_real_np)
        accuracy_2 = self.metrics.ACCURACY_2(yhat_np, y_real_np)
        error_loss = error_loss / len(data_loader)

        return float(error_loss), {'ACCURACY': accuracy, 'ACCURACY_2': accuracy_2}


**Implementando o run.py para realização de um experimento federado**

**1. Upload do Conjunto de Dados:**

* *Arquivo Local:* Se o seu conjunto de dados for um arquivo (por exemplo, CSV, NPZ, etc.), faça o upload para a plataforma e carregue-o usando o caminho ./data/nomedoarquivo.

* *URL:* Se o conjunto de dados estiver disponível em uma URL, inclua a URL no seu código e carregue-o diretamente.

**2. Separação dos dados por cliente:**

* Para simular 4 clientes, divida os dados em 4 partes.

**3. Crie uma instância para FMNISTDataset, PoCModel_2HiddenLayers, PoCExperimentFMNIST.**

**4. Execute as funções:**
* ***generate_server_fn:*** Cria a estratégia para o aprendizado federado
* ***generate_client_fn:*** Gera o modelo e o dataset de cada cliente.
* ***evaluate_fn:*** Avalia o modelo global usando o dataset de um dos clientes.
* ***run_federated:*** Executa o experimento federado.


#### **Modificações para seleção de clientes**
É necessário definir uma classe que herde de uma _strategy_ para ser utilizada ao criar o servidor. 

No exemplo abaixo a ``class MyStrategy(FedAvg)`` poderia ser definida em outro arquivo, por exemplo MyStrategy.py, mas por simplicidade foi declarada no run para evitar imports.

Ao definirmos o servidor utilizamos ``strategy = MyStrategy(...)`` e todo o tratamento da seleção de cliente será feito pela mesma. Nesse exemplo o funcionamento base é:
1. Obtém o número de clientes disponíveis (se todos, K = 100)
2. Calcula a probabilidade de escolher determinado cliente baseado no tamanho do dataset
3. Seleciona os d (30) clientes com maiores probabilidades como candidatos
4. Dentre estes candidatos, solicita os valores de perdas locais
5. Selecione os m (min_fit_clients = 3) com maiores perdas locais e informa para o ClientManager

In [ ]:
from flautim.pytorch.common import run_federated, weighted_average
from flautim.pytorch.federated import Experiment
import TitanicDataset, TitanicModel, TitanicExperiment
from TitanicExperiment import get_params

import flautim as fl
import pandas as pd
import numpy as np
import torch

from datasets import Dataset as HFDataset
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import DirichletPartitioner
from flwr.common import Context, ndarrays_to_parameters, Parameters, FitIns
from flwr.server import ServerConfig, ServerAppComponents
from flwr.server.client_manager import ClientManager
from flwr.server.client_proxy import ClientProxy
from flwr.server.strategy import FedAvg, DifferentialPrivacyServerSideFixedClipping  # <-- DP

# Configuração global
NUM_PARTITIONS = 10

# Pré-processamento do Titanic
def load_and_preprocess():
    titanic = pd.read_csv("./data/Titanic-Dataset.csv")

    titanic['Sex']        = titanic['Sex'].map({'male': 0, 'female': 1})
    titanic['Age']        = titanic['Age'].fillna(titanic['Age'].median())
    titanic['Fare']       = titanic['Fare'].fillna(0.0)
    titanic['Title']      = titanic['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    titanic['Title']      = titanic['Title'].map({'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3}).fillna(4)
    titanic['FamilySize'] = titanic['SibSp'] + titanic['Parch'] + 1
    titanic['IsAlone']    = (titanic['FamilySize'] == 1).astype(int)
    titanic['Embarked']   = titanic['Embarked'].fillna('S').map({'S': 0, 'C': 1, 'Q': 2})
    titanic['Deck']       = titanic['Cabin'].apply(lambda x: str(x)[0] if pd.notna(x) else 'U')
    titanic['Deck']       = pd.factorize(titanic['Deck'])[0]

    titanic = titanic[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
                        'Title', 'FamilySize', 'IsAlone', 'Embarked', 'Deck']]

    return titanic.sample(frac=1, random_state=42).reset_index(drop=True)


# MyStrategy — Power of Choice
class MyStrategy(FedAvg):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.p = []
        self.d = 5
        self.m = self.min_fit_clients

    def configure_fit(
        self,
        server_round: int,
        parameters: Parameters,
        client_manager: ClientManager,
    ) -> list[tuple[ClientProxy, FitIns]]:

        available_clients = list(client_manager.clients.values())
        print(f"Round {server_round} — clientes disponíveis: {[c.cid for c in available_clients]}")

        if self.p == []:
            data_sizes = {
                client.cid: client.fit(
                    FitIns(parameters, {"epochs": -1}),
                    timeout=60,
                    group_id=str(client.cid),
                ).metrics.get("data_size", 0)
                for client in available_clients
            }
            for cid, size in data_sizes.items():
                print(f"  Cliente {cid}: data_size = {size}")

            total = sum(data_sizes.values())
            p_np  = np.array([size / total for size in data_sizes.values()], dtype=float)
            p_np /= p_np.sum()
            p_np[-1] = 1.0 - p_np[:-1].sum()
            self.p = p_np.tolist()

        candidate_clients = np.random.choice(
            available_clients,
            size=min(self.d, len(available_clients)),
            p=self.p,
            replace=False,
        )

        local_losses = {
            client.cid: client.fit(
                FitIns(parameters, {"epochs": 0}),
                timeout=60,
                group_id=str(client.cid),
            ).metrics.get("local_loss", float("inf"))
            for client in candidate_clients
        }

        selected_cids = sorted(local_losses, key=local_losses.get, reverse=True)[: self.m]
        print(f"  Clientes selecionados: {selected_cids}")

        return [
            (client_manager.clients.get(cid), FitIns(parameters, {}))
            for cid in selected_cids
        ]


# Configuração de rodada
def fit_config(server_round: int):
    return {"server_round": server_round}

# Gerador do servidor
def generate_server_fn(context, eval_fn):
    def create_server_fn(context_flwr: Context):
        net               = TitanicModel.TitanicModel(context, num_classes=2, suffix=0)
        global_model_init = ndarrays_to_parameters(get_params(net))

        # Differential Privacy: envolve MyStrategy com DP wrapper
        strategy = DifferentialPrivacyServerSideFixedClipping( 
            strategy=MyStrategy(                                
                evaluate_fn=eval_fn,
                on_fit_config_fn=fit_config,
                on_evaluate_config_fn=fit_config,
                evaluate_metrics_aggregation_fn=weighted_average,
                initial_parameters=global_model_init,
                fraction_fit=0.3,
                min_fit_clients=3,
            ),
            noise_multiplier=0.1,   # ruído gaussiano adicionado aos parâmetros agregados
            clipping_norm=1.0,      # norma máxima dos gradientes antes do ruído
            num_sampled_clients=3,  # deve ser igual a min_fit_clients
        )

        return ServerAppComponents(
            config=ServerConfig(num_rounds=50),
            strategy=strategy,
        )

    return create_server_fn


# Gerador do cliente
def generate_client_fn(context):
    def create_client_fn(context_flwr: Context):
        global fds

        cid       = int(context_flwr.node_config["partition-id"])
        partition = fds.load_partition(cid).train_test_split(test_size=0.2, seed=42)

        model   = TitanicModel.TitanicModel(context, num_classes=2, suffix=cid)
        dataset = TitanicDataset.TitanicDataset(partition, batch_size=10, shuffle=True, num_workers=0)

        return TitanicExperiment.TitanicExperiment(model, dataset, context).to_client()

    return create_client_fn


# Função de avaliação global (executada no servidor)
def evaluate_fn(context):
    def fn(server_round, parameters, config):
        global fds

        model     = TitanicModel.TitanicModel(context, num_classes=2, suffix="FL-Global")
        model.set_parameters(parameters)

        partition  = fds.load_partition(0).train_test_split(test_size=0.2, seed=42)
        dataset    = TitanicDataset.TitanicDataset(partition, batch_size=10, shuffle=False, num_workers=0)
        experiment = TitanicExperiment.TitanicExperiment(model, dataset, context)

        config["server_round"] = server_round
        loss, _, return_dic    = experiment.evaluate(parameters, config)

        return loss, return_dic

    return fn

# Particionamento Non-IID via Dirichlet
def build_federated_dataset(df: pd.DataFrame) -> FederatedDataset:
    hf_dataset  = HFDataset.from_pandas(df)
    partitioner = DirichletPartitioner(
        num_partitions=NUM_PARTITIONS,
        partition_by="Survived",
        alpha=0.3,
        seed=42,
    )
    return FederatedDataset(
        dataset=hf_dataset,
        partitioners={"train": partitioner},
    )

# Entry point
titanic_df = load_and_preprocess()
fds        = build_federated_dataset(titanic_df)

if __name__ == '__main__':
    context = fl.init()
    fl.log("Flautim inicializado!!!")

    client_fn_callback   = generate_client_fn(context)
    evaluate_fn_callback = evaluate_fn(context)
    server_fn_callback   = generate_server_fn(context, eval_fn=evaluate_fn_callback)

    fl.log("Experimento federado Titanic com Differential Privacy criado!")

    run_federated(client_fn_callback, server_fn_callback, num_clients=NUM_PARTITIONS)

#### Referências
- [1] (Flautim tutoriais) Demais Tutorias disponíveis [aqui](https://github.com/FutureLab-DCC/flautim_tutoriais/tree/main)

- [2] (Artigo) Jee Cho, Y., Wang, J., and Joshi, G. (2022). Towards understanding biased client selection in federated learning. In Camps-Valls, G., Ruiz, F. J. R., and Valera, I., editors, Proceedings of The 25th International Conference on Artificial Intelligence and Statistics, volume 151 of Proceedings of Machine Learning Research, pages 10351–10375. PMLR.

- [3] (Discussão no Fórum do Flower - How do I write a custom client selection protocol? & "RepeatHalfSamplingFedAvg") https://discuss.flower.ai/t/how-do-i-write-a-custom-client-selection-protocol/74

- [4] (Discussão no Fórum do Flower - Custom client selection strategy) https://discuss.flower.ai/t/custom-client-selection-strategy/63



<p style="text-align: center;">
✨ Disponibilizado por Carla B. Ferreira, aluna de IC do FutureLab, em 2025. ✨
</p>
